# HydroPulse AI – Exploratory Data Analysis & Sensor Dynamics

**Project**: Intelligent Water Filter Monitoring and Predictive Maintenance System  
**Objective**: Audit the water filter telemetry dataset, analyze sensor degradation trajectories, and inspect physical correlations for dual-target prediction:  
1. **Filter Condition Classification** (`Normal`, `Warning`, `Clogged`)  
2. **Remaining Useful Life (RUL)** Estimation in Days

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150

# Load dataset
dataset_path = os.path.join("..", "dataset", "HydroPulse_synthetic_dataset.csv")
if not os.path.exists(dataset_path):
    dataset_path = r"c:\Users\shyam\Downloads\Atomberg Filter\HydroPulse_synthetic_dataset.csv"

df = pd.read_csv(dataset_path)
print(f"Dataset loaded successfully! Dimensions: {df.shape}")
df.head()

## 1. Dataset Dimensions & Cycle Grouping

Let us examine the filter cycles and samples per cycle. Each filter cycle represents one continuous lifespan of a filter cartridge.

In [ ]:
print("Total Rows:", len(df))
print("Unique Filter Cycles:", df["Filter_Cycle_ID"].nunique())
print("Samples per Filter Cycle:", df["Filter_Cycle_ID"].value_counts().unique())
print("Missing Values:\n", df.isnull().sum())
print("Duplicate Rows:", df.duplicated().sum())

## 2. Target Variable Distributions

Inspect `Filter_Condition` (classification target) and `RUL_days` (regression target).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Condition counts
palette = {"Normal": "#2ca02c", "Warning": "#ff7f0e", "Clogged": "#d62728"}
sns.countplot(x="Filter_Condition", data=df, ax=axes[0], palette=palette, order=["Normal", "Warning", "Clogged"])
axes[0].set_title("Filter Condition Class Distribution (Imbalance Check)", fontweight="bold")

# RUL histogram
sns.histplot(df["RUL_days"], kde=True, ax=axes[1], color="#1f77b4", bins=30)
axes[1].set_title("Remaining Useful Life (RUL) Distribution", fontweight="bold")
axes[1].set_xlabel("RUL (Days)")

plt.tight_layout()
plt.show()

## 3. Physical Degradation Trajectories (Darcy's Law)

As porous membranes accumulate foulants and particulates:
- Head loss rises, causing $\Delta P = P_{in} - P_{out}$ to surge.
- Permeate flow rate decays exponentially or linearly as hydraulic resistance escalates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# DP vs Usage
sns.scatterplot(x="Filter_Usage_days", y="Differential_Pressure_bar", hue="Filter_Condition",
                palette=palette, data=df, alpha=0.7, ax=axes[0])
axes[0].set_title("Differential Pressure vs Filter Usage Days", fontweight="bold")
axes[0].set_xlabel("Usage Days")
axes[0].set_ylabel("Differential Pressure (bar)")

# Flow Rate vs Usage
sns.scatterplot(x="Filter_Usage_days", y="Flow_Rate_L_min", hue="Filter_Condition",
                palette=palette, data=df, alpha=0.7, ax=axes[1])
axes[1].set_title("Flow Rate Decay vs Filter Usage Days", fontweight="bold")
axes[1].set_xlabel("Usage Days")
axes[1].set_ylabel("Flow Rate (L/min)")

plt.tight_layout()
plt.show()

## 4. Feature Correlation Matrix

In [ ]:
features = ["TDS_ppm", "Turbidity_NTU", "Flow_Rate_L_min", "Inlet_Pressure_bar",
            "Outlet_Pressure_bar", "Differential_Pressure_bar", "Filter_Usage_days", "RUL_days"]
corr = df[features].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Sensor Telemetry Correlation Heatmap", fontweight="bold")
plt.tight_layout()
plt.show()